# Khmer OCR testing by image category
Run all, choose Printed Khmer or Khmer handwriting, then click Load selected dataset. Enter an example number from that category’s gallery and click Run real prediction. CPU is sufficient.
These are selected exact-match demo examples, not overall performance on arbitrary images. Printed and handwriting data use different saved checkpoints and preprocessing. The handwriting checkpoint is the available local BiLSTM fine-tune; it is not the Kaggle Transformer specialist.

In [ ]:
%pip -q install torch pillow numpy huggingface_hub ipywidgets
from pathlib import Path
from huggingface_hub import hf_hub_download
import importlib.util,json,zipfile
import ipywidgets as widgets
from IPython.display import display,clear_output
from PIL import Image
CONFIG={
 'printed':{'dataset':'Seoul12/khmer-ocr-selected-demo','archive':'Khmer_OCR_10000_Selected.zip','module':'inference_three.py','class':'MultiRecognizer','checkpoint':'transformer.pt'},
 'handwriting':{'dataset':'Seoul12/khmer-ocr-handwriting-demo','archive':'Handwriting_Khmer_Selected.zip','module':'inference_handwriting.py','class':'HandwritingRecognizer','checkpoint':'handwriting_bilstm.pt'}
}
MODEL_REPO='Seoul12/khmer-ocr-three-architectures'
category=widgets.Dropdown(options=[('Printed Khmer — Transformer','printed'),('Khmer handwriting — BiLSTM fine-tune','handwriting')],description='Category:',layout=widgets.Layout(width='95%'))
load=widgets.Button(description='Load selected dataset',button_style='info')
number=widgets.BoundedIntText(value=1,min=1,max=1,description='Example:')
run=widgets.Button(description='Run real prediction',button_style='primary')
output=widgets.Output();loaded={}
def load_selected(_):
    with output:
        clear_output(wait=True);kind=category.value;cfg=CONFIG[kind]
        print('Loading',kind,'dataset and matching model...')
        archive=hf_hub_download(cfg['dataset'],cfg['archive'],repo_type='dataset')
        base=Path('category_demo')
        with zipfile.ZipFile(archive) as z:z.extractall(base)
        root=base/kind;data=json.loads((root/'manifest.json').read_text())
        module_path=hf_hub_download(MODEL_REPO,cfg['module'])
        spec=importlib.util.spec_from_file_location('ocr_'+kind,module_path);module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
        model=getattr(module,cfg['class'])(hf_hub_download(MODEL_REPO,cfg['checkpoint']))
        loaded.clear();loaded.update(category=kind,root=root,rows=data['images'],model=model)
        number.max=len(data['images']);number.value=1
        print('Ready:',kind,'—',len(data['images']),'selected examples. Choose a number and click Run real prediction.')
        print('Selected demonstration, not overall accuracy on new images.')
def predict(_):
    with output:
        clear_output(wait=True)
        if not loaded or loaded['category']!=category.value:
            print('Click Load selected dataset for this category first.');return
        row=loaded['rows'][number.value-1];path=loaded['root']/row['image']
        display(Image.open(path));prediction=loaded['model'].predict(path)
        print('CATEGORY:',loaded['category'],'EXAMPLE:',number.value)
        print('ANSWER:',row['answer']);print('PREDICTION:',prediction)
        print('RIGHT — exact match' if prediction==row['answer'] else 'WRONG — review input, model and preprocessing')
        print('Selected demonstration example, not overall accuracy.')
load.on_click(load_selected);run.on_click(predict)
display(category,load,number,run,output)
